<a href="https://colab.research.google.com/github/Shokoul/ECE662-Project1/blob/main/ECE662Project1Task3V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch torchvision matplotlib pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import math
import copy
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
from PIL import Image


# ============================================================
# Config
# ============================================================

@dataclass
class CLConfig:
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42

    batch_size: int = 128
    num_workers: int = 2
    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs_per_task: int = 10 #20

    sparse_lambda: float = 1e-4
    recon_lambda: float = 1.0
    protect_lambda: float = 10.0
    orth_lambda: float = 1e-3
    code_l1_lambda: float = 1e-4

    topk_sparsity: int = 8
    grow_threshold: float = 0.25
    grow_atoms: int = 8
    max_atoms: int = 128

    replay_per_task: int = 0  # set >0 if you want tiny replay
    image_size: int = 32

    # dictionary sizes
    dict_atoms_enc1: int = 32
    dict_atoms_enc2: int = 64
    dict_atoms_fc: int = 64
    dict_atoms_dec1: int = 64
    dict_atoms_dec2: int = 32

    latent_dim: int = 128
    num_classes: int = 10

    save_dir: str = "./outputs"


# ============================================================
# Utilities
# ============================================================

def set_seed(seed: int) -> None:
    import random
    import numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def compute_psnr_from_mse(mse: torch.Tensor, max_val: float = 1.0) -> torch.Tensor:
    eps = 1e-12
    return 10.0 * torch.log10((max_val ** 2) / (mse + eps))


def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    pred = logits.argmax(dim=1)
    return (pred == y).float().mean().item()


def count_nonzero_tensor(t: torch.Tensor) -> int:
    return int((t != 0).sum().item())


# ============================================================
# Dataset helpers
# ============================================================

class SynthDigitsFolder(Dataset):
    """
    Expected folder structure:
      root/
        train/
          0/xxx.png
          1/yyy.png
          ...
        test/
          0/zzz.png
          ...
    """

    def __init__(self, root: str, split: str = "train", transform=None):
        self.transform = transform
        self.samples = []
        split_root = os.path.join(root, split)
        if not os.path.isdir(split_root):
            raise FileNotFoundError(f"SynthDigits path not found: {split_root}")

        for cls_name in sorted(os.listdir(split_root)):
            cls_path = os.path.join(split_root, cls_name)
            if not os.path.isdir(cls_path):
                continue
            if not cls_name.isdigit():
                continue
            label = int(cls_name)
            for fname in os.listdir(cls_path):
                if fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                    self.samples.append((os.path.join(cls_path, fname), label))

        if len(self.samples) == 0:
            raise RuntimeError(f"No images found in {split_root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, y = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, y


def build_transforms(dataset_name: str, image_size: int = 32):
    if dataset_name.lower() in ["mnist", "usps"]:
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
        ])
    else:
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])
    return transform


def get_dataset(dataset_name: str, root: str = "./data", split: str = "train", image_size: int = 32):
    name = dataset_name.lower()
    transform = build_transforms(name, image_size)

    if name == "mnist":
        return datasets.MNIST(root=root, train=(split == "train"), download=True, transform=transform)
    if name == "usps":
        return datasets.USPS(root=root, train=(split == "train"), download=True, transform=transform)
    if name == "svhn":
        svhn_split = "train" if split == "train" else "test"
        return datasets.SVHN(root=root, split=svhn_split, download=True, transform=transform)
    if name == "cifar10":
        return datasets.CIFAR10(root=root, train=(split == "train"), download=True, transform=transform)
    if name == "synthdigits":
        synth_root = os.path.join(root, "synthdigits")
        return SynthDigitsFolder(synth_root, split=split, transform=transform)

    raise ValueError(f"Unsupported dataset: {dataset_name}")


def get_targets(dataset) -> List[int]:
    if hasattr(dataset, "targets"):
        targets = dataset.targets
        if isinstance(targets, list):
            return targets
        return list(targets)
    if hasattr(dataset, "labels"):
        labels = dataset.labels
        if isinstance(labels, list):
            return labels
        return list(labels)
    raise AttributeError("Dataset has no targets or labels attribute.")


def filter_dataset_by_classes(dataset, classes: List[int]) -> Subset:
    targets = get_targets(dataset)
    idxs = [i for i, y in enumerate(targets) if int(y) in classes]
    return Subset(dataset, idxs)


def make_class_incremental_tasks(
    dataset_name: str,
    root: str,
    image_size: int,
    task_splits: List[List[int]]
) -> Tuple[List[Subset], List[Subset]]:
    train_ds = get_dataset(dataset_name, root=root, split="train", image_size=image_size)
    test_ds = get_dataset(dataset_name, root=root, split="test", image_size=image_size)

    train_tasks = [filter_dataset_by_classes(train_ds, cls_group) for cls_group in task_splits]
    test_tasks = [filter_dataset_by_classes(test_ds, cls_group) for cls_group in task_splits]
    return train_tasks, test_tasks


def make_domain_incremental_tasks(
    dataset_names: List[str],
    root: str,
    image_size: int
) -> Tuple[List[Dataset], List[Dataset]]:
    train_tasks = [get_dataset(name, root=root, split="train", image_size=image_size) for name in dataset_names]
    test_tasks = [get_dataset(name, root=root, split="test", image_size=image_size) for name in dataset_names]
    return train_tasks, test_tasks


# ============================================================
# Sparse code helpers
# ============================================================

def topk_mask(x: torch.Tensor, k: int) -> torch.Tensor:
    """
    Keep only top-k absolute entries along channel/features dimension.
    x: [B, C, ...]
    """
    if k <= 0 or k >= x.shape[1]:
        return torch.ones_like(x)

    with torch.no_grad():
        flat = x.abs().flatten(start_dim=2).mean(dim=2) if x.dim() > 2 else x.abs()
        vals, idx = torch.topk(flat, k=k, dim=1)
        mask = torch.zeros_like(flat)
        mask.scatter_(1, idx, 1.0)

        if x.dim() > 2:
            for _ in range(x.dim() - 2):
                mask = mask.unsqueeze(-1)
        return mask


def hard_topk_sparse(x: torch.Tensor, k: int) -> torch.Tensor:
    mask = topk_mask(x, k)
    return x * mask


# ============================================================
# Dictionary-based layers
# ============================================================

class SparseDictConv2d(nn.Module):
    """
    Shared atoms + task-specific codes for conv filters.

    Weight construction:
        W_task = sum_j alpha_task[j] * atom_j
    where atoms are shared and alpha_task are task-specific trainable codes.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        num_atoms: int,
        stride: int = 1,
        padding: int = 0,
        bias: bool = True,
        topk: int = 8,
        task_id: int = 0,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.num_atoms = num_atoms
        self.stride = stride
        self.padding = padding
        self.topk = topk

        # shared atoms: [K, out, in, k, k]
        self.atoms = nn.Parameter(
            0.02 * torch.randn(num_atoms, out_channels, in_channels, kernel_size, kernel_size)
        )

        # task-specific codes stored in dict-like structure
        self.task_codes = nn.ParameterDict()
        self.task_biases = nn.ParameterDict() if bias else None
        self.register_buffer("importance", torch.zeros(num_atoms))
        self.add_task(task_id)

    def add_task(self, task_id: int):
        key = str(task_id)
        if key not in self.task_codes:
            code = nn.Parameter(0.02 * torch.randn(self.num_atoms))
            self.task_codes[key] = code
            if self.task_biases is not None:
                self.task_biases[key] = nn.Parameter(torch.zeros(self.out_channels))

    def maybe_grow(self, task_id: int, n_new: int, max_atoms: int):
        if self.num_atoms >= max_atoms or n_new <= 0:
            return

        n_new = min(n_new, max_atoms - self.num_atoms)
        if n_new <= 0:
            return

        device = self.atoms.device
        new_atoms = 0.02 * torch.randn(
            n_new, self.out_channels, self.in_channels, self.kernel_size, self.kernel_size, device=device
        )
        with torch.no_grad():
            old_atoms = self.atoms.data
            grown = torch.cat([old_atoms, new_atoms], dim=0)

        self.num_atoms = grown.shape[0]
        self.atoms = nn.Parameter(grown)

        # grow all existing task codes with zeros
        old_codes = {}
        for key, p in self.task_codes.items():
            old_codes[key] = p.data.clone()

        self.task_codes = nn.ParameterDict()
        for key, old in old_codes.items():
            new_code = torch.zeros(self.num_atoms, device=device)
            new_code[: old.shape[0]] = old
            self.task_codes[key] = nn.Parameter(new_code)

        if self.task_biases is not None:
            old_biases = {}
            for key, p in self.task_biases.items():
                old_biases[key] = p.data.clone()
            self.task_biases = nn.ParameterDict()
            for key, b in old_biases.items():
                self.task_biases[key] = nn.Parameter(b)

        old_importance = self.importance.data.clone()
        new_importance = torch.zeros(self.num_atoms, device=device)
        new_importance[: old_importance.shape[0]] = old_importance
        self.register_buffer("importance", new_importance)

    def get_sparse_code(self, task_id: int) -> torch.Tensor:
        code = self.task_codes[str(task_id)]
        return hard_topk_sparse(code.unsqueeze(0), self.topk).squeeze(0)

    def build_weight(self, task_id: int) -> torch.Tensor:
        code = self.get_sparse_code(task_id)  # [K]
        weight = torch.einsum("k,koihw->oihw", code, self.atoms)
        return weight

    def forward(self, x: torch.Tensor, task_id: int) -> torch.Tensor:
        weight = self.build_weight(task_id)
        bias = self.task_biases[str(task_id)] if self.task_biases is not None else None
        return F.conv2d(x, weight, bias=bias, stride=self.stride, padding=self.padding)

    def l1_code(self, task_id: int) -> torch.Tensor:
        return self.get_sparse_code(task_id).abs().sum()

    def orth_loss(self) -> torch.Tensor:
        flat = self.atoms.view(self.num_atoms, -1)
        gram = flat @ flat.t()
        eye = torch.eye(self.num_atoms, device=flat.device)
        return ((gram - eye) ** 2).mean()

    def protect_loss(self, old_atoms: torch.Tensor) -> torch.Tensor:
        # importance-weighted atom protection
        imp = self.importance.view(-1, 1, 1, 1, 1)
        return ((imp * (self.atoms - old_atoms)) ** 2).mean()

    def update_importance(self, task_id: int):
        with torch.no_grad():
            code = self.get_sparse_code(task_id).abs()
            self.importance[: code.shape[0]] = torch.maximum(self.importance[: code.shape[0]], code.detach())


class SparseDictLinear(nn.Module):
    """
    Shared atoms + task-specific codes for linear weights.
    Weight:
        W_task = sum_j alpha_task[j] * atom_j
    """

    def __init__(self, in_features: int, out_features: int, num_atoms: int, bias: bool = True, topk: int = 8, task_id: int = 0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.num_atoms = num_atoms
        self.topk = topk

        self.atoms = nn.Parameter(0.02 * torch.randn(num_atoms, out_features, in_features))
        self.task_codes = nn.ParameterDict()
        self.task_biases = nn.ParameterDict() if bias else None
        self.register_buffer("importance", torch.zeros(num_atoms))
        self.add_task(task_id)

    def add_task(self, task_id: int):
        key = str(task_id)
        if key not in self.task_codes:
            self.task_codes[key] = nn.Parameter(0.02 * torch.randn(self.num_atoms))
            if self.task_biases is not None:
                self.task_biases[key] = nn.Parameter(torch.zeros(self.out_features))

    def maybe_grow(self, task_id: int, n_new: int, max_atoms: int):
        if self.num_atoms >= max_atoms or n_new <= 0:
            return

        n_new = min(n_new, max_atoms - self.num_atoms)
        if n_new <= 0:
            return

        device = self.atoms.device
        new_atoms = 0.02 * torch.randn(n_new, self.out_features, self.in_features, device=device)
        with torch.no_grad():
            old_atoms = self.atoms.data
            grown = torch.cat([old_atoms, new_atoms], dim=0)

        self.num_atoms = grown.shape[0]
        self.atoms = nn.Parameter(grown)

        old_codes = {}
        for key, p in self.task_codes.items():
            old_codes[key] = p.data.clone()

        self.task_codes = nn.ParameterDict()
        for key, old in old_codes.items():
            new_code = torch.zeros(self.num_atoms, device=device)
            new_code[: old.shape[0]] = old
            self.task_codes[key] = nn.Parameter(new_code)

        if self.task_biases is not None:
            old_biases = {}
            for key, p in self.task_biases.items():
                old_biases[key] = p.data.clone()
            self.task_biases = nn.ParameterDict()
            for key, b in old_biases.items():
                self.task_biases[key] = nn.Parameter(b)

        old_importance = self.importance.data.clone()
        new_importance = torch.zeros(self.num_atoms, device=device)
        new_importance[: old_importance.shape[0]] = old_importance
        self.register_buffer("importance", new_importance)

    def get_sparse_code(self, task_id: int) -> torch.Tensor:
        code = self.task_codes[str(task_id)]
        return hard_topk_sparse(code.unsqueeze(0), self.topk).squeeze(0)

    def build_weight(self, task_id: int) -> torch.Tensor:
        code = self.get_sparse_code(task_id)
        weight = torch.einsum("k,koi->oi", code, self.atoms)
        return weight

    def forward(self, x: torch.Tensor, task_id: int) -> torch.Tensor:
        weight = self.build_weight(task_id)
        bias = self.task_biases[str(task_id)] if self.task_biases is not None else None
        return F.linear(x, weight, bias)

    def l1_code(self, task_id: int) -> torch.Tensor:
        return self.get_sparse_code(task_id).abs().sum()

    def orth_loss(self) -> torch.Tensor:
        flat = self.atoms.view(self.num_atoms, -1)
        gram = flat @ flat.t()
        eye = torch.eye(self.num_atoms, device=flat.device)
        return ((gram - eye) ** 2).mean()

    def protect_loss(self, old_atoms: torch.Tensor) -> torch.Tensor:
        imp = self.importance.view(-1, 1, 1)
        return ((imp * (self.atoms - old_atoms)) ** 2).mean()

    def update_importance(self, task_id: int):
        with torch.no_grad():
            code = self.get_sparse_code(task_id).abs()
            self.importance[: code.shape[0]] = torch.maximum(self.importance[: code.shape[0]], code.detach())


# ============================================================
# Continual model
# ============================================================

class SCDCLNet(nn.Module):
    """
    Sparse Coding / Dictionary Learning Continual Learning model.
    Supports classification + reconstruction.
    """

    def __init__(self, cfg: CLConfig):
        super().__init__()
        self.cfg = cfg

        self.enc1 = SparseDictConv2d(3, 32, kernel_size=3, stride=1, padding=1,
                                     num_atoms=cfg.dict_atoms_enc1, topk=min(cfg.topk_sparsity, cfg.dict_atoms_enc1), task_id=0)
        self.enc2 = SparseDictConv2d(32, 64, kernel_size=3, stride=1, padding=1,
                                     num_atoms=cfg.dict_atoms_enc2, topk=min(cfg.topk_sparsity, cfg.dict_atoms_enc2), task_id=0)

        self.pool = nn.MaxPool2d(2, 2)
        self.flatten_dim = 64 * (cfg.image_size // 4) * (cfg.image_size // 4)

        self.fc_latent = SparseDictLinear(self.flatten_dim, cfg.latent_dim,
                                          num_atoms=cfg.dict_atoms_fc, topk=min(cfg.topk_sparsity, cfg.dict_atoms_fc), task_id=0)

        # global classifier head, fixed to 10 classes by default
        self.cls_head = nn.Linear(cfg.latent_dim, cfg.num_classes)

        # lightweight decoder for reconstruction
        self.dec_fc = nn.Linear(cfg.latent_dim, self.flatten_dim)
        self.dec1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = nn.ConvTranspose2d(32, 3, kernel_size=2, stride=2)

        self.task_ids = {0}

    def add_task(self, task_id: int):
        self.task_ids.add(task_id)
        self.enc1.add_task(task_id)
        self.enc2.add_task(task_id)
        self.fc_latent.add_task(task_id)

    def encode(self, x: torch.Tensor, task_id: int) -> torch.Tensor:
        x = F.relu(self.enc1(x, task_id))
        x = self.pool(x)
        x = F.relu(self.enc2(x, task_id))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        z = F.relu(self.fc_latent(x, task_id))
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.dec_fc(z))
        x = x.view(x.size(0), 64, self.cfg.image_size // 4, self.cfg.image_size // 4)
        x = F.relu(self.dec1(x))
        x = torch.sigmoid(self.dec2(x))
        return x

    def forward(self, x: torch.Tensor, task_id: int):
        z = self.encode(x, task_id)
        logits = self.cls_head(z)
        recon = self.decode(z)
        return logits, recon

    def sparse_regularization(self, task_id: int) -> torch.Tensor:
        return (
            self.enc1.l1_code(task_id)
            + self.enc2.l1_code(task_id)
            + self.fc_latent.l1_code(task_id)
        )

    def orth_regularization(self) -> torch.Tensor:
        return (
            self.enc1.orth_loss()
            + self.enc2.orth_loss()
            + self.fc_latent.orth_loss()
        )

    def snapshot_shared_atoms(self) -> Dict[str, torch.Tensor]:
        return {
            "enc1": self.enc1.atoms.detach().clone(),
            "enc2": self.enc2.atoms.detach().clone(),
            "fc_latent": self.fc_latent.atoms.detach().clone(),
        }

    def protection_loss(self, snapshot: Dict[str, torch.Tensor]) -> torch.Tensor:
        return (
            self.enc1.protect_loss(snapshot["enc1"])
            + self.enc2.protect_loss(snapshot["enc2"])
            + self.fc_latent.protect_loss(snapshot["fc_latent"])
        )

    def update_importance(self, task_id: int):
        self.enc1.update_importance(task_id)
        self.enc2.update_importance(task_id)
        self.fc_latent.update_importance(task_id)

    def maybe_grow(self, task_id: int, n_new: int):
        self.enc1.maybe_grow(task_id, n_new, self.cfg.max_atoms)
        self.enc2.maybe_grow(task_id, n_new, self.cfg.max_atoms)
        self.fc_latent.maybe_grow(task_id, n_new, self.cfg.max_atoms)

    def memory_footprint_bytes(self) -> int:
        """
        Rough memory estimate:
        shared atoms + all task codes + decoder/classifier dense params
        """
        total_params = 0

        # shared atoms
        total_params += self.enc1.atoms.numel()
        total_params += self.enc2.atoms.numel()
        total_params += self.fc_latent.atoms.numel()

        # task-specific codes and biases
        for p in self.enc1.task_codes.values():
            total_params += count_nonzero_tensor(hard_topk_sparse(p.unsqueeze(0), self.enc1.topk).squeeze(0))
        for p in self.enc2.task_codes.values():
            total_params += count_nonzero_tensor(hard_topk_sparse(p.unsqueeze(0), self.enc2.topk).squeeze(0))
        for p in self.fc_latent.task_codes.values():
            total_params += count_nonzero_tensor(hard_topk_sparse(p.unsqueeze(0), self.fc_latent.topk).squeeze(0))

        if self.enc1.task_biases is not None:
            for p in self.enc1.task_biases.values():
                total_params += p.numel()
        if self.enc2.task_biases is not None:
            for p in self.enc2.task_biases.values():
                total_params += p.numel()
        if self.fc_latent.task_biases is not None:
            for p in self.fc_latent.task_biases.values():
                total_params += p.numel()

        # dense shared modules
        total_params += sum(p.numel() for p in self.cls_head.parameters())
        total_params += sum(p.numel() for p in self.dec_fc.parameters())
        total_params += sum(p.numel() for p in self.dec1.parameters())
        total_params += sum(p.numel() for p in self.dec2.parameters())

        return int(total_params * 4)  # float32 approx


# ============================================================
# Replay buffer (optional)
# ============================================================

class TinyReplayBuffer:
    def __init__(self, max_per_task: int = 0):
        self.max_per_task = max_per_task
        self.data: Dict[int, List[Tuple[torch.Tensor, int]]] = {}

    def add_from_loader(self, task_id: int, loader: DataLoader):
        if self.max_per_task <= 0:
            return
        if task_id in self.data:
            return

        samples = []
        for x, y in loader:
            for i in range(x.size(0)):
                samples.append((x[i].clone(), int(y[i])))
                if len(samples) >= self.max_per_task:
                    self.data[task_id] = samples
                    return
        self.data[task_id] = samples

    def get_batch(self, batch_size: int, device: str):
        if self.max_per_task <= 0 or len(self.data) == 0:
            return None, None

        xs = []
        ys = []
        per_task = max(1, batch_size // len(self.data))
        for _, items in self.data.items():
            chosen = items[:per_task]
            for x, y in chosen:
                xs.append(x)
                ys.append(y)

        if len(xs) == 0:
            return None, None

        x = torch.stack(xs, dim=0).to(device)
        y = torch.tensor(ys, dtype=torch.long, device=device)
        return x, y


# ============================================================
# Metrics
# ============================================================

def evaluate_task(model: SCDCLNet, loader: DataLoader, task_id: int, device: str):
    model.eval()
    total_acc = 0.0
    total_mse = 0.0
    total_n = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits, recon = model(x, task_id)
            batch_acc = accuracy_from_logits(logits, y)
            mse = F.mse_loss(recon, x, reduction="mean")
            total_acc += batch_acc * x.size(0)
            total_mse += mse.item() * x.size(0)
            total_n += x.size(0)

    avg_acc = total_acc / max(total_n, 1)
    avg_mse = total_mse / max(total_n, 1)
    psnr = compute_psnr_from_mse(torch.tensor(avg_mse)).item()
    return {
        "accuracy": avg_acc,
        "mse": avg_mse,
        "psnr": psnr,
    }


def compute_forgetting(acc_matrix: List[List[float]]) -> List[float]:
    """
    acc_matrix[t][i] = accuracy on task i after learning task t
    """
    forgetting = []
    for t in range(len(acc_matrix)):
        if t == 0:
            forgetting.append(0.0)
            continue
        vals = []
        for i in range(t):
            best_prev = max(acc_matrix[k][i] for k in range(i, t))
            current = acc_matrix[t][i]
            vals.append(best_prev - current)
        forgetting.append(sum(vals) / max(len(vals), 1))
    return forgetting


# ============================================================
# Training
# ============================================================

def build_optimizer(model: SCDCLNet, cfg: CLConfig):
    return torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)


def train_one_task(
    model: SCDCLNet,
    train_loader: DataLoader,
    task_id: int,
    cfg: CLConfig,
    replay: Optional[TinyReplayBuffer] = None,
):
    model.train()
    device = cfg.device
    optimizer = build_optimizer(model, cfg)

    atom_snapshot = model.snapshot_shared_atoms() if task_id > 0 else None
    epoch_logs = []

    for epoch in range(cfg.epochs_per_task):
        running = {
            "loss": 0.0,
            "cls": 0.0,
            "rec": 0.0,
            "sparse": 0.0,
            "protect": 0.0,
            "orth": 0.0,
            "acc": 0.0,
            "n": 0,
        }

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            if replay is not None and cfg.replay_per_task > 0:
                rx, ry = replay.get_batch(batch_size=max(8, x.size(0) // 4), device=device)
                if rx is not None:
                    x = torch.cat([x, rx], dim=0)
                    y = torch.cat([y, ry], dim=0)

            optimizer.zero_grad()

            logits, recon = model(x, task_id)

            cls_loss = F.cross_entropy(logits, y)
            rec_loss = F.mse_loss(recon, x)
            sparse_loss = model.sparse_regularization(task_id)
            orth_loss = model.orth_regularization()

            if atom_snapshot is not None:
                protect_loss = model.protection_loss(atom_snapshot)
            else:
                protect_loss = torch.tensor(0.0, device=device)

            loss = (
                cls_loss
                + cfg.recon_lambda * rec_loss
                + cfg.code_l1_lambda * sparse_loss
                + cfg.protect_lambda * protect_loss
                + cfg.orth_lambda * orth_loss
            )

            loss.backward()
            optimizer.step()

            with torch.no_grad():
                acc = accuracy_from_logits(logits, y)

            running["loss"] += loss.item() * x.size(0)
            running["cls"] += cls_loss.item() * x.size(0)
            running["rec"] += rec_loss.item() * x.size(0)
            running["sparse"] += sparse_loss.item() * x.size(0)
            running["protect"] += protect_loss.item() * x.size(0)
            running["orth"] += orth_loss.item() * x.size(0)
            running["acc"] += acc * x.size(0)
            running["n"] += x.size(0)

        epoch_log = {k: (v / max(running["n"], 1)) for k, v in running.items() if k != "n"}
        epoch_log["epoch"] = epoch + 1
        epoch_logs.append(epoch_log)
        print(
            f"[Task {task_id}][Epoch {epoch+1}/{cfg.epochs_per_task}] "
            f"loss={epoch_log['loss']:.4f} "
            f"cls={epoch_log['cls']:.4f} "
            f"rec={epoch_log['rec']:.4f} "
            f"acc={epoch_log['acc']:.4f}"
        )

    model.update_importance(task_id)
    return epoch_logs


def maybe_expand_dictionary(
    model: SCDCLNet,
    val_loader: DataLoader,
    task_id: int,
    cfg: CLConfig
):
    metrics = evaluate_task(model, val_loader, task_id, cfg.device)
    # simple heuristic: if cls error high, grow
    cls_error = 1.0 - metrics["accuracy"]
    if cls_error > cfg.grow_threshold:
        print(f"Growing dictionary for task {task_id}: cls_error={cls_error:.4f}")
        model.maybe_grow(task_id, cfg.grow_atoms)
    else:
        print(f"No growth needed for task {task_id}: cls_error={cls_error:.4f}")


# ============================================================
# Continual runner
# ============================================================

def make_loader(dataset, cfg: CLConfig, shuffle: bool = True) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )


def run_continual_experiment(
    train_tasks: List[Dataset],
    test_tasks: List[Dataset],
    cfg: CLConfig,
    experiment_name: str = "class_incremental",
):
    ensure_dir(cfg.save_dir)
    model = SCDCLNet(cfg).to(cfg.device)
    replay = TinyReplayBuffer(cfg.replay_per_task)

    acc_matrix: List[List[float]] = []
    psnr_matrix: List[List[float]] = []
    memory_curve: List[float] = []
    all_logs = []

    for task_id, train_task in enumerate(train_tasks):
        print(f"\n====================")
        print(f"Training task {task_id}")
        print(f"====================")

        if task_id > 0:
            model.add_task(task_id)

        train_loader = make_loader(train_task, cfg, shuffle=True)

        # grow before or after short warmup; here before full training by quick eval is skipped
        epoch_logs = train_one_task(model, train_loader, task_id, cfg, replay=replay)

        # optional growth after task training based on own task performance
        maybe_expand_dictionary(model, train_loader, task_id, cfg)

        # small replay buffer
        replay.add_from_loader(task_id, train_loader)

        # evaluate on all seen tasks
        seen_accs = []
        seen_psnrs = []
        for eval_task_id in range(task_id + 1):
            test_loader = make_loader(test_tasks[eval_task_id], cfg, shuffle=False)
            metrics = evaluate_task(model, test_loader, eval_task_id, cfg.device)
            seen_accs.append(metrics["accuracy"])
            seen_psnrs.append(metrics["psnr"])
            print(
                f"Eval after task {task_id} on task {eval_task_id}: "
                f"acc={metrics['accuracy']:.4f}, psnr={metrics['psnr']:.2f}"
            )

        acc_matrix.append(seen_accs + [0.0] * (len(train_tasks) - len(seen_accs)))
        psnr_matrix.append(seen_psnrs + [0.0] * (len(train_tasks) - len(seen_psnrs)))

        mem_mb = model.memory_footprint_bytes() / (1024 ** 2)
        memory_curve.append(mem_mb)

        all_logs.append({
            "task_id": task_id,
            "epoch_logs": epoch_logs,
            "seen_accs": seen_accs,
            "seen_psnrs": seen_psnrs,
            "memory_mb": mem_mb,
        })

    forgetting_curve = compute_forgetting(acc_matrix)

    result = {
        "experiment_name": experiment_name,
        "acc_matrix": acc_matrix,
        "psnr_matrix": psnr_matrix,
        "memory_curve_mb": memory_curve,
        "forgetting_curve": forgetting_curve,
        "logs": all_logs,
    }

    out_path = os.path.join(cfg.save_dir, f"{experiment_name}_results.json")
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"\nSaved results to: {out_path}")
    return result, model


# ============================================================
# Plotting
# ============================================================

def plot_curves(result: Dict, save_dir: str):
    import matplotlib.pyplot as plt

    ensure_dir(save_dir)
    acc_matrix = result["acc_matrix"]
    psnr_matrix = result["psnr_matrix"]
    memory_curve = result["memory_curve_mb"]
    forgetting_curve = result["forgetting_curve"]

    num_tasks = len(acc_matrix)
    xs = list(range(1, num_tasks + 1))

    avg_acc = []
    avg_psnr = []
    for t in range(num_tasks):
        seen = t + 1
        avg_acc.append(sum(acc_matrix[t][:seen]) / seen)
        avg_psnr.append(sum(psnr_matrix[t][:seen]) / seen)

    plt.figure(figsize=(7, 5))
    plt.plot(xs, avg_acc, marker="o")
    plt.xlabel("Number of tasks learned")
    plt.ylabel("Average accuracy")
    plt.title("Accuracy vs Number of Tasks")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "accuracy_vs_tasks_V2.png"))
    plt.close()

    plt.figure(figsize=(7, 5))
    plt.plot(xs, avg_psnr, marker="o")
    plt.xlabel("Number of tasks learned")
    plt.ylabel("Average PSNR")
    plt.title("PSNR vs Number of Tasks")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "psnr_vs_tasks_V2.png"))
    plt.close()

    plt.figure(figsize=(7, 5))
    plt.plot(xs, memory_curve, marker="o")
    plt.xlabel("Number of tasks learned")
    plt.ylabel("Memory footprint (MB)")
    plt.title("Memory Footprint vs Number of Tasks")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "memory_vs_tasks_V2.png"))
    plt.close()

    plt.figure(figsize=(7, 5))
    plt.plot(xs, forgetting_curve, marker="o")
    plt.xlabel("Task index")
    plt.ylabel("Average forgetting")
    plt.title("Forgetting Curve")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "forgetting_curve_v2.png"))
    plt.close()


# ============================================================
# Task builders
# ============================================================

def build_class_incremental_digit_tasks(root: str, dataset_name: str, image_size: int = 32):
    task_splits = [
        [0, 1],
        [2, 3],
        [4, 5],
        [6, 7],
        [8, 9],
    ]
    return make_class_incremental_tasks(
        dataset_name=dataset_name,
        root=root,
        image_size=image_size,
        task_splits=task_splits,
    )


def build_class_incremental_cifar_tasks(root: str, image_size: int = 32):
    task_splits = [
        [0, 1],
        [2, 3],
        [4, 5],
        [6, 7],
        [8, 9],
    ]
    return make_class_incremental_tasks(
        dataset_name="cifar10",
        root=root,
        image_size=image_size,
        task_splits=task_splits,
    )


def build_domain_incremental_digit_tasks(root: str, image_size: int = 32):
    dataset_order = ["mnist", "usps", "svhn", "synthdigits"]
    return make_domain_incremental_tasks(dataset_order, root=root, image_size=image_size)


# ============================================================
# Main examples
# ============================================================

def main():
    cfg = CLConfig()
    set_seed(cfg.seed)
    ensure_dir(cfg.save_dir)

    # --------------------------------------------------------
    # Example 1: MNIST class-incremental
    # --------------------------------------------------------
    mnist_train_tasks, mnist_test_tasks = build_class_incremental_digit_tasks(
        root="./data",
        dataset_name="mnist",
        image_size=cfg.image_size
    )

    result_mnist, model_mnist = run_continual_experiment(
        train_tasks=mnist_train_tasks,
        test_tasks=mnist_test_tasks,
        cfg=cfg,
        experiment_name="mnist_class_incremental"
    )
    plot_curves(result_mnist, os.path.join(cfg.save_dir, "mnist_class_incremental"))

    # --------------------------------------------------------
    # Example 2: USPS class-incremental
    # --------------------------------------------------------
    usps_train_tasks, usps_test_tasks = build_class_incremental_digit_tasks(
        root="./data",
        dataset_name="usps",
        image_size=cfg.image_size
    )

    result_usps, _ = run_continual_experiment(
        train_tasks=usps_train_tasks,
        test_tasks=usps_test_tasks,
        cfg=cfg,
        experiment_name="usps_class_incremental"
    )
    plot_curves(result_usps, os.path.join(cfg.save_dir, "usps_class_incremental"))

    # --------------------------------------------------------
    # Example 3: SVHN class-incremental
    # --------------------------------------------------------
    svhn_train_tasks, svhn_test_tasks = build_class_incremental_digit_tasks(
        root="./data",
        dataset_name="svhn",
        image_size=cfg.image_size
    )

    result_svhn, _ = run_continual_experiment(
        train_tasks=svhn_train_tasks,
        test_tasks=svhn_test_tasks,
        cfg=cfg,
        experiment_name="svhn_class_incremental"
    )
    plot_curves(result_svhn, os.path.join(cfg.save_dir, "svhn_class_incremental"))

    # --------------------------------------------------------
    # Example 4: CIFAR10 class-incremental
    # --------------------------------------------------------
    cifar_train_tasks, cifar_test_tasks = build_class_incremental_cifar_tasks(
        root="./data",
        image_size=cfg.image_size
    )

    result_cifar, _ = run_continual_experiment(
        train_tasks=cifar_train_tasks,
        test_tasks=cifar_test_tasks,
        cfg=cfg,
        experiment_name="cifar10_class_incremental"
    )
    plot_curves(result_cifar, os.path.join(cfg.save_dir, "cifar10_class_incremental"))

    # --------------------------------------------------------
    # Example 5: Domain-incremental digits
    # SynthDigits must exist under ./data/synthdigits/{train,test}/{0..9}/...
    # --------------------------------------------------------
    try:
        domain_train_tasks, domain_test_tasks = build_domain_incremental_digit_tasks(
            root="./data",
            image_size=cfg.image_size
        )

        result_domain, _ = run_continual_experiment(
            train_tasks=domain_train_tasks,
            test_tasks=domain_test_tasks,
            cfg=cfg,
            experiment_name="digits_domain_incremental"
        )
        plot_curves(result_domain, os.path.join(cfg.save_dir, "digits_domain_incremental"))
    except Exception as e:
        print(f"Skipping domain-incremental experiment: {e}")


if __name__ == "__main__":
    main()

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 435kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.07MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.9MB/s]



Training task 0


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[Task 0][Epoch 1/10] loss=0.5638 cls=0.3965 rec=0.0932 acc=0.8547
[Task 0][Epoch 2/10] loss=0.0538 cls=0.0064 rec=0.0395 acc=0.9981
[Task 0][Epoch 3/10] loss=0.0387 cls=0.0070 rec=0.0265 acc=0.9973
[Task 0][Epoch 4/10] loss=0.0291 cls=0.0042 rec=0.0211 acc=0.9987
[Task 0][Epoch 5/10] loss=0.0272 cls=0.0053 rec=0.0188 acc=0.9983
[Task 0][Epoch 6/10] loss=0.0238 cls=0.0038 rec=0.0168 acc=0.9988
[Task 0][Epoch 7/10] loss=0.0206 cls=0.0036 rec=0.0142 acc=0.9991
[Task 0][Epoch 8/10] loss=0.0232 cls=0.0059 rec=0.0145 acc=0.9983
[Task 0][Epoch 9/10] loss=0.0201 cls=0.0044 rec=0.0124 acc=0.9980
[Task 0][Epoch 10/10] loss=0.0210 cls=0.0060 rec=0.0120 acc=0.9976
No growth needed for task 0: cls_error=0.0026
Eval after task 0 on task 0: acc=0.9972, psnr=19.69

Training task 1
[Task 1][Epoch 1/10] loss=1.2347 cls=1.1393 rec=0.0704 acc=0.5071
[Task 1][Epoch 2/10] loss=0.1814 cls=0.1192 rec=0.0489 acc=0.9576
[Task 1][Epoch 3/10] loss=0.1344 cls=0.0837 rec=0.0425 acc=0.9708
[Task 1][Epoch 4/10] loss=

100%|██████████| 6.58M/6.58M [00:01<00:00, 4.11MB/s]
100%|██████████| 1.83M/1.83M [00:00<00:00, 2.49MB/s]


...


Training task 0


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[Task 0][Epoch 1/10] loss=0.8510 cls=0.7667 rec=0.0416 acc=0.6961
[Task 0][Epoch 2/10] loss=0.6085 cls=0.5788 rec=0.0263 acc=0.7369
[Task 0][Epoch 3/10] loss=0.5949 cls=0.5739 rec=0.0184 acc=0.7359
[Task 0][Epoch 4/10] loss=0.5754 cls=0.5571 rec=0.0153 acc=0.7311
[Task 0][Epoch 5/10] loss=0.5611 cls=0.5423 rec=0.0145 acc=0.7335
[Task 0][Epoch 6/10] loss=0.4894 cls=0.4666 rec=0.0148 acc=0.7709
[Task 0][Epoch 7/10] loss=0.3191 cls=0.2878 rec=0.0158 acc=0.8809
[Task 0][Epoch 8/10] loss=0.2447 cls=0.2143 rec=0.0129 acc=0.9175
[Task 0][Epoch 9/10] loss=0.2048 cls=0.1763 rec=0.0109 acc=0.9349
[Task 0][Epoch 10/10] loss=0.1861 cls=0.1591 rec=0.0103 acc=0.9417
No growth needed for task 0: cls_error=0.0711
Eval after task 0 on task 0: acc=0.9421, psnr=19.45

Training task 1
[Task 1][Epoch 1/10] loss=0.9062 cls=0.8639 rec=0.0328 acc=0.4897
[Task 1][Epoch 2/10] loss=0.7079 cls=0.6884 rec=0.0178 acc=0.5505
[Task 1][Epoch 3/10] loss=0.6902 cls=0.6723 rec=0.0151 acc=0.5747
[Task 1][Epoch 4/10] loss=

100%|██████████| 170M/170M [00:03<00:00, 47.8MB/s]



Training task 0
[Task 0][Epoch 1/10] loss=1.1046 cls=0.9491 rec=0.0690 acc=0.5065
[Task 0][Epoch 2/10] loss=0.7396 cls=0.6601 rec=0.0640 acc=0.6043
[Task 0][Epoch 3/10] loss=0.6860 cls=0.6209 rec=0.0543 acc=0.6655
[Task 0][Epoch 4/10] loss=0.6552 cls=0.6022 rec=0.0427 acc=0.6766
[Task 0][Epoch 5/10] loss=0.6150 cls=0.5664 rec=0.0387 acc=0.7092
[Task 0][Epoch 6/10] loss=0.5903 cls=0.5411 rec=0.0390 acc=0.7268
[Task 0][Epoch 7/10] loss=0.5573 cls=0.5085 rec=0.0379 acc=0.7479
[Task 0][Epoch 8/10] loss=0.5342 cls=0.4879 rec=0.0352 acc=0.7581
[Task 0][Epoch 9/10] loss=0.5051 cls=0.4606 rec=0.0327 acc=0.7798
[Task 0][Epoch 10/10] loss=0.5045 cls=0.4611 rec=0.0321 acc=0.7768
No growth needed for task 0: cls_error=0.2045
Eval after task 0 on task 0: acc=0.8030, psnr=14.62

Training task 1
[Task 1][Epoch 1/10] loss=1.0558 cls=0.9736 rec=0.0676 acc=0.4597
[Task 1][Epoch 2/10] loss=0.7372 cls=0.6861 rec=0.0468 acc=0.5586
[Task 1][Epoch 3/10] loss=0.7180 cls=0.6742 rec=0.0405 acc=0.5741
[Task 1][